In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

# ALL paths and adjustable experiment settings live here; hide this cell if needed.
from pathlib import Path

PROJECT_DIR = PROJECT_DIR
DATA_DIR = PROCESSED_DIR
WEATHER_DIR = WEATHER_DIR
ZONE_DIR = ZONE_DIR
HELPER_DIR = PROJECT_DIR / "src" / "forecasting"
TEST_PICKUP_DIR = (
    DATA_DIR / "model_evaluation_2026" / "fhvhv_by_zone_hour_2026_01_01_to_07.parquet"
)
RUN_NAME = (
    "zonal_week_ahead_v1"  # New settings -> new run name; preserve earlier results.
)
RESULTS_DIR = PROJECT_DIR / "outputs" / "06_zonal_forecasting" / RUN_NAME
# Reproducibility and bounded sample sizes; these do not start a run.
SEED = 42
MAX_TRAIN_EXAMPLES = (
    100_000  # Seeded sample spanning ALL training years; adjust for available RAM.
)
SHAP_ROWS = 100
SHAP_BACKGROUND = 30
# Time boundaries are local New York clock times.
TRAIN_START = "2021-01-01 00:00"
TRAIN_END = "2026-01-01 00:00"  # Exclusive for labels.
TEST_ORIGIN = "2026-01-01 00:00"  # New York time, fixed for the entire week.
FOLDS = ["2025-04-01 00:00", "2025-07-01 00:00", "2025-10-01 00:00", "2025-12-01 00:00"]
# ONE execution mode controls the sequential workflow below.
# review: definitions only; features: generate/export data; full: train and evaluate.
EXECUTION_MODE = "review"  # Change to features or full deliberately.
assert EXECUTION_MODE in {"review", "features", "full"}
FEATURE_DIR = RESULTS_DIR / "features"
NOTEBOOK_PATH = PROJECT_DIR / "notebooks" / "06_model_hourly_demand_forecasting.ipynb"
N_CANDIDATES_PER_MODEL = 6
MAX_ABLATION_GROUPS = 2
ABLATION_GROUPS = []  # Empty -> test the two lowest-ranked nonessential SHAP groups.
FAIL_ON_FIT_WARNINGS = False  # Warnings are always saved; True makes review mandatory.
ACKNOWLEDGE_FIT_WARNINGS = False
SAVE_FIGURES = True

RUN_DATA_AUDIT = EXECUTION_MODE in {"features", "full"}
RUN_FEATURE_EXPORT = EXECUTION_MODE in {"features", "full"}
RUN_CV = EXECUTION_MODE == "full"
RUN_SHAP = EXECUTION_MODE == "full"
RUN_ABLATION = EXECUTION_MODE == "full"
RUN_FINAL = EXECUTION_MODE == "full"


# 06 — End-to-end zonal hourly pickup forecasting

Forecast **January 1–7, 2026 (168 hours per eligible zone)** using 2021–2025 history. This notebook contains the actual feature-engineering, estimator, fitting, tuning and selection code, rather than only calls to hidden implementations.

## How to run
1. Save the notebook, select the Python (llm) kernel and install missing dependencies using the commented setup command if needed.
2. Set **`EXECUTION_MODE = "features"`** and Run All to create inspectable training/validation feature datasets without fitting models.
3. Set **`EXECUTION_MODE = "full"`** and Run All to generate/reuse features, tune Random Forest/XGBoost/neural-network models and ARIMA, select by chronological validation, run SHAP/ablation, freeze settings, refit and evaluate the test week.
4. Leave **`EXECUTION_MODE = "review"`** to inspect code without loading the full data or running models. This is the saved default; no training was run while authoring.

The full run is computationally expensive on a laptop: six sampled configurations per ML family, three ARIMA orders, four validation weeks, then SHAP/ablation and final fitting. Model fits run sequentially. Compatible completed candidate/fold predictions and feature bundles can be reused on rerun. Change `RUN_NAME` when changing code, inputs or experiment settings. The five training-history windows remain in notebook 07.

## 1. Feature list — visible without running the notebook

**40 candidate predictors before encoding:** 6 categorical variables are one-hot encoded; the remaining 34 are numerical or binary. The pickup-count target and timestamps are not predictor columns.

| Role | Variable | Definition |
|---|---|---|
| Target | `pickup_count` | Recorded pickups in one taxi zone and target hour. |
| Metadata | `target_datetime` | Local hour being predicted; not passed directly to the model. |
| Metadata | `origin` | Issue time shared by the 168 predictions for a forecast week. |

### Zone identity
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `zone_id` | Taxi-zone LocationID; an identifier, not a numerical ordering. | category | one-hot |
| `borough` | Borough recorded in the taxi-zone shapefile; Unknown when unmapped. | category | one-hot |

### Spatial and data-quality features
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `is_airport_zone` | 1 for EWR (1), JFK (132), or LaGuardia (138); otherwise 0. | 0/1 | numeric |
| `zone_latitude` | Latitude of a representative point within the zone geometry. | degrees north | numeric |
| `zone_longitude` | Longitude of the same representative point. | degrees east | numeric |
| `zone_area_km2` | Zone polygon area calculated in a projected coordinate system. | km² | numeric |
| `zone_unmapped` | 1 when the zone has no matching shapefile geometry. | 0/1 | numeric |

### Calendar features
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `forecast_horizon` | Hours ahead of issue time; horizon 1 is the first forecast hour. | 1–168 | numeric |
| `hour` | Local clock hour of the target timestamp. | 0–23 | one-hot |
| `day_of_week` | Target weekday: Monday=0 through Sunday=6. | 0–6 | one-hot |
| `month` | Calendar month of the target timestamp. | 1–12 | one-hot |
| `hour_of_week` | Target weekday × 24 + target hour. | 0–167 | numeric |
| `is_weekend` | 1 when the target falls on Saturday or Sunday. | 0/1 | numeric |
| `hour_sin` | sin(2π × target hour / 24). | unitless | numeric |
| `hour_cos` | cos(2π × target hour / 24). | unitless | numeric |
| `annual_sin` | Sine of the annual cycle, accounting for leap years. | unitless | numeric |
| `annual_cos` | Cosine of the annual cycle, accounting for leap years. | unitless | numeric |

### Holiday features
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `is_holiday` | 1 on a selected US federal actual or observed holiday date. | 0/1 | numeric |
| `holiday_name` | Federal holiday name for the target date, or none. | category | one-hot |
| `is_observed_holiday` | 1 on a shifted observed date that differs from the actual holiday date. | 0/1 | numeric |
| `days_to_holiday` | Calendar days until the next selected holiday, including 0 today. | days | numeric |
| `days_since_holiday` | Calendar days since the previous selected holiday, including 0 today. | days | numeric |
| `is_year_end_period` | 1 during the predefined December 24–January 2 period. | 0/1 | numeric |

### Target-aligned demand history
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `pickup_lag_168h` | Same-zone pickups 168 clock hours before the target hour. | trips/hour | numeric |
| `pickup_lag_336h` | Same-zone pickups 336 clock hours before the target hour. | trips/hour | numeric |
| `pickup_lag_504h` | Same-zone pickups 504 clock hours before the target hour. | trips/hour | numeric |
| `same_hour_week_mean_4w` | Mean pickups at target lags 168, 336, 504 and 672 hours. | trips/hour | numeric |

### Origin-based recent demand
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `recent_mean_7d` | Zone mean over 7 days strictly before the forecast origin; fixed across the week. | trips/hour | numeric |
| `recent_mean_28d` | Zone mean over 28 days strictly before the forecast origin. | trips/hour | numeric |
| `recent_std_28d` | Zone standard deviation over those same pre-origin 28 days. | trips/hour | numeric |
| `recent_trend_ratio` | Latest pre-origin 7-day mean divided by the preceding 7-day mean; missing if denominator=0. | ratio | numeric |

### Weather features
| Variable | Definition | Unit | Encoding |
|---|---|---|---|
| `temperature_2m` | Past-only mean air temperature for the target month/hour. | °C | numeric |
| `apparent_temperature` | Past-only mean feels-like temperature for the target month/hour. | °C | numeric |
| `relative_humidity_2m` | Past-only mean relative humidity for the target month/hour. | % | numeric |
| `precipitation` | Past-only mean hourly total precipitation for the target month/hour. | mm/hour | numeric |
| `rain` | Past-only mean hourly liquid precipitation for the target month/hour. | mm/hour | numeric |
| `snowfall` | Past-only mean hourly snowfall for the target month/hour. | cm/hour | numeric |
| `wind_speed_10m` | Past-only mean wind speed at 10 m for the target month/hour. | m/s | numeric |
| `cloud_cover` | Past-only mean cloud cover for the target month/hour. | % | numeric |
| `pressure_msl` | Past-only mean sea-level pressure for the target month/hour. | hPa | numeric |

**Availability:** demand predictors use information strictly before the forecast origin. Recent summaries stay fixed across that week. Weather predictors use past-only climatology, not actual weather during the forecast week. See the feature-engineering section for handling missing values, zeros and DST.
**Feature selection:** retain the full candidate set initially. Use validation SHAP plus retraining to decide removals; never choose features from final-test results.

## Environment setup
Core feature/model functions are defined explicitly below. The external helper is now used for the data dictionary and plotting utilities only. No installation or fitting happens automatically in review mode.

In [ ]:
# Import dependencies and report the active kernel before starting any work.
# Install only if needed in THIS notebook kernel; commands are comments, not automatic.
# %pip install "xgboost>=2.1,<3" "shap>=0.46,<0.47"
# Base: Python 3.9, numpy 2.0.2, pandas 2.3.3, sklearn 1.6.1, statsmodels 0.14.6.
# Also required: duckdb, geopandas, matplotlib and a Parquet engine (pyarrow).
import sys, json, hashlib, warnings, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(HELPER_DIR))
import zonal_forecasting as zf

print("Kernel executable:", sys.executable)
versions = zf.package_versions()
display(pd.Series(versions, name="version").to_frame())
missing = [p for p in ["xgboost", "shap"] if importlib.util.find_spec(p) is None]
if missing:
    print("Install before modeling/explanation stages:", ", ".join(missing))

# Imports used by the visible implementations below.
import copy
import time
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import ParameterSampler
from statsmodels.tsa.arima.model import ARIMA

if EXECUTION_MODE == "full" and missing:
    raise ImportError(
        "Install the listed XGBoost/SHAP dependencies in this kernel first."
    )


In [ ]:
# Predictor groups control encoding, SHAP regrouping and group ablation consistently.
HOURS = 168
CAT = list(zf.CAT)
GROUPS = copy.deepcopy(zf.GROUPS)
FEATURES = [name for columns in GROUPS.values() for name in columns]
WEATHER = GROUPS["weather"]


### Small persistence utility

In [ ]:
def json_write(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")


### Handle local clock exceptions

In [ ]:
def clock_valid(index):
    """Mask nonexistent spring and combined autumn clock hours explicitly."""
    return (
        ~pd.DatetimeIndex(index)
        .tz_localize("America/New_York", ambiguous="NaT", nonexistent="NaT")
        .isna()
    )


### Generate holiday flags and distances

In [ ]:
def holiday_features(index):
    """US federal actual/observed dates; predefined Christmas-New Year period Dec24-Jan2."""
    dates = pd.DatetimeIndex(index).normalize()
    low = dates.min() - pd.Timedelta(days=400)
    high = dates.max() + pd.Timedelta(days=400)
    calendar = USFederalHolidayCalendar()
    observed = calendar.holidays(low, high, return_name=True)
    parts = []
    for rule in calendar.rules:
        actual = copy.copy(rule)
        actual.observance = None
        parts.append(actual.dates(low, high, return_name=True))
    actual = pd.concat(parts).sort_index()
    actual = actual[~actual.index.duplicated()]
    observed = observed[~observed.index.duplicated()]
    union = actual.index.union(observed.index).sort_values()
    ticks = union.asi8
    nxt = np.searchsorted(ticks, dates.asi8, side="left")
    prev = np.searchsorted(ticks, dates.asi8, side="right") - 1
    names = (
        pd.Series(dates)
        .map(actual)
        .astype("string")
        .fillna(pd.Series(dates).map(observed).astype("string"))
        .fillna("none")
    )
    return pd.DataFrame(
        dict(
            is_holiday=dates.isin(union).astype(int),
            holiday_name=names.to_numpy(),
            is_observed_holiday=(
                dates.isin(observed.index) & ~dates.isin(actual.index)
            ).astype(int),
            days_to_holiday=(ticks[nxt] - dates.asi8) / 86400e9,
            days_since_holiday=(dates.asi8 - ticks[prev]) / 86400e9,
            is_year_end_period=(
                ((dates.month == 12) & (dates.day >= 24))
                | ((dates.month == 1) & (dates.day <= 2))
            ).astype(int),
        )
    )


### Read zone geography and derive spatial features

In [ ]:
def load_static(cfg):
    """Consolidate duplicate zone polygons, then calculate spatial predictors.

    Area is calculated in a projected CRS; representative points lie inside polygons.
    Boroughs come from the shapefile rather than being inferred from zone ID numbers.
    """
    import geopandas as gpd

    files = list(Path(cfg["ZONE_DIR"]).glob("*.shp"))
    assert len(files) == 1
    zones = gpd.read_file(files[0])
    assert zones.groupby("LocationID").borough.nunique().le(1).all()
    zones = zones[["LocationID", "borough", "geometry"]].dissolve(
        by="LocationID", as_index=False, aggfunc="first"
    )
    projected = zones.to_crs(2263)
    points = projected.geometry.representative_point().to_crs(4326)
    info = pd.DataFrame(
        dict(
            zone_id=zones.LocationID.astype(int),
            borough=zones.borough,
            zone_latitude=points.y,
            zone_longitude=points.x,
            zone_area_km2=projected.geometry.area * 0.3048006096012192**2 / 1e6,
        )
    )
    info["is_airport_zone"] = info.zone_id.isin([1, 132, 138]).astype(
        int
    )  # EWR, JFK, LGA
    info["zone_unmapped"] = 0
    return info.set_index("zone_id"), zones


### Read and validate an existing monthly pickup aggregate

In [ ]:
def read_month(folder, year, month, db, allow_partial=False):
    """Audit existing aggregates before interpreting sparse absent zone keys as zero."""
    folder = Path(folder)
    files = sorted(folder.glob("*.parquet"))
    assert (
        files and (folder / "_SUCCESS").exists()
    ), f"Missing/incomplete pickup aggregate: {folder}"
    if year in (2021, 2022, 2023):
        c = json.loads(
            (folder / "_aggregation_contract.json").read_text(encoding="utf-8")
        )
        assert (c["version"], c["family"], c["year"], c["month"]) == (
            "earlier-years-hourly-v2",
            "pickup",
            year,
            month,
        )
    db.read_parquet([str(f) for f in files]).create_view("source", replace=True)
    bad = db.execute(
        f"""SELECT count(*) FROM source WHERE PULocationID IS NULL OR date IS NULL
 OR hour IS NULL OR hour NOT BETWEEN 0 AND 23 OR trip_count IS NULL OR trip_count<=0
 OR year(date)!={year} OR month(date)!={month}"""
    ).fetchone()[0]
    duplicate = db.execute(
        "SELECT count(*) FROM (SELECT 1 FROM source GROUP BY PULocationID,date,hour HAVING count(*)>1)"
    ).fetchone()[0]
    assert bad == duplicate == 0, f"Invalid values/duplicate keys: {folder.name}"
    data = db.execute(
        "SELECT PULocationID AS zone_id,date,hour,trip_count AS pickups FROM source"
    ).df()
    data["timestamp"] = pd.to_datetime(data.date) + pd.to_timedelta(data.hour, unit="h")
    if not allow_partial:
        start = pd.Timestamp(year=year, month=month, day=1)
        expected = pd.date_range(
            start, start + pd.offsets.MonthBegin(1), freq="h", inclusive="left"
        )
        assert (
            expected[clock_valid(expected)].isin(data.timestamp.unique()).all()
        ), f"Missing citywide hours: {folder.name}"
    return data[["zone_id", "timestamp", "pickups"]], files


## Visible feature generation
The following cells perform the actual calculations used for both exported datasets and model fitting. Small mixin classes organize related methods so no single cell contains the entire feature pipeline.

In [ ]:
class HistorySummaryMixin:
    def prepare_recent_history(self):
        # Shift by one hour first: every origin summary excludes that origin's target.
        prior = self.panel.shift(1)
        self.mean7 = (
            prior.rolling(168, min_periods=144).mean().to_numpy(dtype="float32")
        )
        self.mean28 = (
            prior.rolling(672, min_periods=600).mean().to_numpy(dtype="float32")
        )
        self.std28 = (
            prior.rolling(672, min_periods=600).std(ddof=0).to_numpy(dtype="float32")
        )
        previous7 = (
            prior.shift(168)
            .rolling(168, min_periods=144)
            .mean()
            .to_numpy(dtype="float32")
        )
        self.trend = np.divide(
            self.mean7,
            previous7,
            out=np.full_like(self.mean7, np.nan),
            where=previous7 > 0,
        )


In [ ]:
class WeatherInputMixin:
    def prepare_weather_inputs(self):
        # Actual weather is retained for diagnostics. Prediction features use past-only cache lookups.
        self.weather = pd.read_parquet(
            Path(self.cfg["WEATHER_DIR"]) / "nyc_hourly_weather_local_2021_2026.parquet"
        )
        self.weather["timestamp"] = pd.to_datetime(self.weather.date) + pd.to_timedelta(
            self.weather.hour, unit="h"
        )
        assert not self.weather.duplicated(["weather_location_id", "timestamp"]).any()
        mapping = pd.read_csv(
            Path(self.cfg["WEATHER_DIR"]) / "taxi_zone_weather_lookup.csv"
        ).set_index("LocationID")
        self.weather_locations = (
            mapping.reindex(self.zone_ids)
            .weather_location_id.fillna("manhattan")
            .to_numpy()
        )
        self.weather_proxy = pd.Series(
            ~pd.Index(self.zone_ids).isin(mapping.index), index=self.zone_ids
        )
        self._build_weather_cache()

    def _build_weather_cache(self):
        """Build cumulative weather sums so each example can query its own past only."""
        self.weather_cache = {}
        for location, g in self.weather.groupby("weather_location_id"):
            for month in range(13):
                for hour in range(24) if month else [-1]:
                    selected = (
                        g
                        if month == 0
                        else g[
                            (g.timestamp.dt.month == month)
                            & (g.timestamp.dt.hour == hour)
                        ]
                    )
                    selected = selected.sort_values("timestamp")
                    v = selected[WEATHER].to_numpy(float)
                    assert np.isfinite(v).all()
                    self.weather_cache[(location, month, hour)] = (
                        selected.timestamp.to_numpy("datetime64[ns]"),
                        np.vstack([np.zeros(len(WEATHER)), np.cumsum(v, axis=0)]),
                    )


### Estimate future weather using past-only climatology
Every lookup excludes weather at or after that example’s own issue time. These estimated predictors must not be confused with actual weather in the forecast week.

In [ ]:
class WeatherClimatologyMixin:
    def climatology(self, locations, targets, origins):
        """Prefix sums exclude all weather at/after each row's own forecast origin."""
        result = np.full((len(targets), len(WEATHER)), np.nan)
        q = pd.DataFrame(
            dict(location=locations, month=targets.month, hour=targets.hour)
        )
        issue = origins.to_numpy("datetime64[ns]")
        start = np.datetime64(self.cfg["TRAIN_START"])
        for (location, month, hour), indices in q.groupby(
            ["location", "month", "hour"]
        ).groups.items():
            rows = np.asarray(indices)
            dates, sums = self.weather_cache[(location, int(month), int(hour))]
            lo = np.searchsorted(dates, start, side="left")
            counts = np.searchsorted(dates, issue[rows], side="left")
            n = counts - lo
            good = n > 0
            result[rows[good]] = (sums[counts[good]] - sums[lo]) / n[good, None]
            if (~good).any():
                dates, sums = self.weather_cache[(location, 0, -1)]
                lo = np.searchsorted(dates, start, side="left")
                r = rows[~good]
                counts = np.searchsorted(dates, issue[r], side="left")
                n = counts - lo
                assert (n > 0).all(), "No pre-origin weather history"
                result[r] = (sums[counts] - sums[lo]) / n[:, None]
        return result


In [ ]:
class PanelAccessMixin:
    def positions(self, times):
        return self.index.get_indexer(pd.DatetimeIndex(times))

    def values_at(self, times, zone_indices):
        position = self.positions(times)
        result = np.full(len(position), np.nan)
        valid = position >= 0
        result[valid] = self.values[position[valid], np.asarray(zone_indices)[valid]]
        return result

    def scales(self, cutoff):
        train = self.panel[self.panel.index < pd.Timestamp(cutoff)]
        scale = (train - train.shift(168)).abs().mean()
        return scale.where(scale > 0)

    def observations(self, meta):
        return self.values_at(
            meta.target_datetime, pd.Index(self.zone_ids).get_indexer(meta.zone_id)
        )


### Calendar features: values known before the forecast week

In [ ]:
def make_calendar_features(targets, horizons):
    x = pd.DataFrame(
        dict(
            forecast_horizon=horizons,
            hour=targets.hour,
            day_of_week=targets.dayofweek,
            month=targets.month,
            hour_of_week=targets.dayofweek * 24 + targets.hour,
            is_weekend=(targets.dayofweek >= 5).astype(int),
            hour_sin=np.sin(2 * np.pi * targets.hour / 24),
            hour_cos=np.cos(2 * np.pi * targets.hour / 24),
            annual_sin=np.sin(
                2
                * np.pi
                * (targets.dayofyear - 1)
                / np.where(targets.is_leap_year, 366, 365)
            ),
            annual_cos=np.cos(
                2
                * np.pi
                * (targets.dayofyear - 1)
                / np.where(targets.is_leap_year, 366, 365)
            ),
        )
    )
    x = pd.concat([x, holiday_features(targets)], axis=1)
    return x


### Historical-demand features: target-aligned weekly lags and origin summaries

In [ ]:
def make_demand_features(self, targets, origins, zone_indices):
    x = pd.DataFrame(index=range(len(targets)))
    origin_pos = self.positions(origins)
    # Use target-aligned WEEKLY lags; short lags would require unknown within-week demand.
    for lag in (168, 336, 504):
        x[f"pickup_lag_{lag}h"] = self.values_at(
            targets - pd.Timedelta(hours=lag), zone_indices
        )
    lag4 = np.column_stack(
        [
            self.values_at(targets - pd.Timedelta(hours=lag), zone_indices)
            for lag in (168, 336, 504, 672)
        ]
    )
    x["same_hour_week_mean_4w"] = np.mean(
        lag4, axis=1
    )  # DST missingness remains explicit.
    # Origin summaries are fixed across the week, unlike target-aligned weekly lags.
    for name, array in [
        ("recent_mean_7d", self.mean7),
        ("recent_mean_28d", self.mean28),
        ("recent_std_28d", self.std28),
        ("recent_trend_ratio", self.trend),
    ]:
        x[name] = array[origin_pos, zone_indices]
    return x


### Combine calendar, demand, zone and weather predictors
This returns the real 40-column feature matrix plus separate metadata. One-hot encoding is fitted later on training rows only.

In [ ]:
class FeatureMatrixMixin:
    def features(self, origins, zone_indices, horizons):
        """One direct target per row. All demand inputs precede the row's origin."""
        origins = pd.DatetimeIndex(origins)
        zone_indices = np.asarray(zone_indices, dtype=int)
        horizons = np.asarray(horizons, dtype=int)
        assert ((horizons >= 1) & (horizons <= 168)).all()
        targets = origins + pd.to_timedelta(horizons - 1, unit="h")
        # Even the last target's one-week lag must precede the forecast origin.
        assert ((targets - pd.Timedelta(hours=168)) < origins).all()
        origin_pos = self.positions(origins)
        assert (origin_pos >= 0).all()
        # Calendar values are known in advance for every target hour.
        x = make_calendar_features(targets, horizons)
        demand = make_demand_features(self, targets, origins, zone_indices)
        x = pd.concat([x, demand], axis=1)
        static = self.static.iloc[zone_indices].reset_index()
        for name in ["zone_id"] + GROUPS["spatial"] + ["borough"]:
            x[name] = static[name].to_numpy()
        w = self.climatology(self.weather_locations[zone_indices], targets, origins)
        for i, name in enumerate(WEATHER):
            x[name] = w[:, i]
        # Store categories as strings so zone IDs/hours are never treated as ordered numbers.
        for name in CAT:
            x[name] = x[name].astype(str)
        meta = pd.DataFrame(
            dict(
                zone_id=self.zone_ids[zone_indices],
                origin=origins,
                target_datetime=targets,
                forecast_horizon=horizons,
                forecast_day=(horizons - 1) // 24 + 1,
            )
        )
        return x[FEATURES], meta

    def forecast_examples(self, origin):
        issues = pd.DatetimeIndex([pd.Timestamp(origin)] * (len(self.zone_ids) * 168))
        return self.features(
            issues,
            np.repeat(np.arange(len(self.zone_ids)), 168),
            np.tile(np.arange(1, 169), len(self.zone_ids)),
        )


In [ ]:
class TrainingExamplesMixin:
    def generate_training_examples(self, cutoff):
        """Reproducible bounded sample covering all years; same examples for all ML candidates.
        Sampled zone/day-origin/horizon triples avoid materializing every overlapping week.
        All labels, including hour 168, must be strictly before the fold cutoff.
        """
        cutoff = pd.Timestamp(cutoff)
        rng = np.random.default_rng(self.cfg["SEED"])
        origins = pd.date_range(
            pd.Timestamp(self.cfg["TRAIN_START"]) + pd.Timedelta(days=28),
            cutoff - pd.Timedelta(days=7),
            freq="D",
        )
        assert len(origins) > 0
        n = self.cfg["MAX_TRAIN_EXAMPLES"]
        triples = pd.DataFrame(
            dict(
                o=rng.integers(len(origins), size=n),
                z=rng.integers(len(self.zone_ids), size=n),
                h=rng.integers(1, 169, size=n),
            )
        ).drop_duplicates()
        x, meta = self.features(
            origins.take(triples.o), triples.z.to_numpy(), triples.h.to_numpy()
        )
        y = self.values_at(meta.target_datetime, triples.z.to_numpy())
        keep = np.isfinite(y) & clock_valid(meta.target_datetime)
        assert (meta.target_datetime < cutoff).all()
        x = x.loc[keep].reset_index(drop=True)
        meta = meta.loc[keep].reset_index(drop=True)
        y = y[keep]
        assert len(y) > 100
        return x, y, meta

    def training_examples(self, cutoff):
        """Read the exact exported training matrix when available; otherwise generate it."""
        folder = feature_bundle_folder(self, cutoff)
        receipt = folder / "feature_manifest.json"
        if receipt.exists():
            assert json.loads(receipt.read_text()) == feature_bundle_signature(
                self, cutoff
            )
            x = pd.read_parquet(folder / "train_features.parquet")
            labeled = pd.read_parquet(folder / "train_targets_and_metadata.parquet")
            assert (labeled.target_datetime < pd.Timestamp(cutoff)).all()
            y = labeled.pop("pickup_count").to_numpy(float)
            return x, y, labeled
        return self.generate_training_examples(cutoff)


In [ ]:
class AuditExportMixin:
    def save_audit(self, manifest, excluded, unknown, index):
        root = Path(self.cfg["RESULTS_DIR"])
        json_write(root / "input_manifest.json", manifest)
        pd.DataFrame(
            dict(
                zone_id=self.zone_ids,
                weather_location_id=self.weather_locations,
                uses_city_weather_proxy=self.weather_proxy.to_numpy(),
                unmapped_geometry=unknown.to_numpy(),
            )
        ).to_csv(root / "zone_eligibility.csv", index=False)
        json_write(
            root / "data_audit.json",
            dict(
                training_shape=self.values.shape,
                eligible_zones=self.zone_ids.tolist(),
                excluded_zero_activity_zones=excluded,
                dst_clock_hours_excluded=int((~clock_valid(index)).sum()),
                zero_policy="Absent zone keys are zero only within audited complete citywide months; DST hours excluded.",
                weather_policy="Climatology from TRAIN_START strictly before each example origin.",
            ),
        )


### Assemble the data context
The constructor audits the source, prepares recent-history statistics and weather lookups, and fixes zone eligibility. It does not fit models.

In [ ]:
class ForecastData(
    HistorySummaryMixin,
    WeatherInputMixin,
    WeatherClimatologyMixin,
    PanelAccessMixin,
    FeatureMatrixMixin,
    TrainingExamplesMixin,
    AuditExportMixin,
):
    def __init__(self, cfg):
        import duckdb

        self.cfg = cfg
        self.static, self.zones = load_static(cfg)
        # Read monthly aggregates sequentially to bound memory; do not load raw trips.
        db = duckdb.connect()
        db.execute("SET threads=1")
        db.execute("SET memory_limit='256MB'")
        # Include one empty origin row at TRAIN_END so shifted summaries exist for Jan 1.
        index = pd.date_range(
            cfg["TRAIN_START"], cfg["TRAIN_END"], freq="h", inclusive="both"
        )
        panel = pd.DataFrame(0, index=index, columns=self.static.index, dtype="float32")
        manifest = []
        for year in range(2021, 2026):
            for month in range(1, 13):
                folder = (
                    Path(cfg["DATA_DIR"])
                    / f"fhvhv_by_zone_hour_{year}_{month:02d}.parquet"
                )
                data, files = read_month(folder, year, month, db)
                wide = data.pivot(
                    index="timestamp", columns="zone_id", values="pickups"
                )
                for zone in wide.columns.difference(panel.columns):
                    panel[int(zone)] = np.float32(0)
                panel.loc[wide.index, wide.columns] = wide.fillna(0).astype("float32")
                manifest.append(
                    dict(
                        year=year,
                        month=month,
                        path=str(folder),
                        parts=len(files),
                        rows=len(data),
                        source_parts=[
                            dict(
                                name=f.name,
                                size=f.stat().st_size,
                                mtime_ns=f.stat().st_mtime_ns,
                            )
                            for f in files
                        ],
                    )
                )
        db.close()
        panel = panel.sort_index(axis=1).astype("float32")
        panel.loc[~clock_valid(index), :] = np.nan
        panel.loc[pd.Timestamp(cfg["TRAIN_END"]), :] = np.nan
        # Freeze zone eligibility using observations BEFORE the first validation week.
        eligible = panel.loc[panel.index < pd.Timestamp(cfg["FOLDS"][0])].sum().gt(0)
        excluded = panel.columns[~eligible].tolist()
        self.panel = panel.loc[:, eligible]
        self.zone_ids = self.panel.columns.to_numpy(dtype=int)
        self.index = self.panel.index
        self.values = self.panel.to_numpy(dtype="float32")
        self.static = self.static.reindex(self.zone_ids)
        unknown = self.static.borough.isna()
        self.static.loc[unknown, "borough"] = "Unknown"
        self.static.loc[unknown, "is_airport_zone"] = (
            self.static.index[unknown].isin([1, 132, 138]).astype(int)
        )
        self.static["zone_unmapped"] = unknown.astype(int)
        self.prepare_recent_history()
        self.manifest = manifest
        self.prepare_weather_inputs()
        self.save_audit(manifest, excluded, unknown, index)


## Generate and export feature data independently of modeling
The exported Parquet tables are the training inputs actually read by the tuning code. Feature columns, labels and metadata are saved separately. Validation labels are never included as predictors; final January labels are not read here.

In [ ]:
def feature_bundle_folder(data, cutoff):
    key = pd.Timestamp(cutoff).strftime("origin_%Y%m%d")
    return Path(data.cfg["FEATURE_DIR"]) / key


def feature_bundle_signature(data, cutoff):
    """Bind a feature bundle to the code, data sources and sampling settings."""
    manifest = Path(data.cfg["RESULTS_DIR"]) / "input_manifest.json"
    return {
        "origin": str(pd.Timestamp(cutoff)),
        "seed": data.cfg["SEED"],
        "max_train_examples": data.cfg["MAX_TRAIN_EXAMPLES"],
        "implementation_sha256": data.cfg["IMPLEMENTATION_SHA256"],
        "source_manifest_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(),
        "feature_columns": FEATURES,
    }


def export_feature_bundle(data, cutoff, include_validation_labels):
    """Create reusable feature tables without fitting any estimator."""
    folder = feature_bundle_folder(data, cutoff)
    folder.mkdir(parents=True, exist_ok=True)
    signature = feature_bundle_signature(data, cutoff)
    receipt = folder / "feature_manifest.json"
    if receipt.exists():
        assert (
            json.loads(receipt.read_text()) == signature
        ), "Use a new RUN_NAME for changed inputs."
        print("Reusing feature bundle:", folder.name)
        return

    # Training labels end before the forecast origin, including for 168-hour examples.
    x_train, y_train, train_metadata = data.generate_training_examples(cutoff)
    train_metadata = train_metadata.assign(pickup_count=y_train)
    x_future, future_metadata = data.forecast_examples(cutoff)
    assert list(x_train.columns) == list(x_future.columns) == FEATURES
    assert (train_metadata.target_datetime < pd.Timestamp(cutoff)).all()

    x_train.to_parquet(folder / "train_features.parquet", index=False)
    train_metadata.to_parquet(
        folder / "train_targets_and_metadata.parquet", index=False
    )
    x_future.to_parquet(folder / "forecast_features.parquet", index=False)
    if include_validation_labels:
        future_metadata = future_metadata.assign(
            pickup_count=data.observations(future_metadata)
        )
    future_metadata.to_parquet(folder / "forecast_metadata.parquet", index=False)

    # Missing predictors are retained for train-only imputation; missing labels are not fabricated.
    quality = {
        "training_rows": len(x_train),
        "forecast_rows": len(x_future),
        "predictor_columns": len(x_train.columns),
        "training_missing_features": x_train.isna().sum().to_dict(),
        "includes_validation_labels": include_validation_labels,
    }
    json_write(folder / "feature_quality.json", quality)
    json_write(receipt, signature)  # Written last, after all feature files exist.
    print("Exported", folder.name, ":", len(x_train), "training rows")


## Visible model pipeline and evaluation functions
These definitions create reusable estimators. Fitting occurs only in the explicitly enabled execution stages later in the notebook.

In [ ]:
def make_preprocessor(columns):
    """Learn imputation, scaling and categories from training rows only."""
    cats = [c for c in CAT if c in columns]
    numeric = [c for c in columns if c not in cats]
    preprocess = ColumnTransformer(
        [
            (
                "num",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", add_indicator=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore", sparse_output=False, dtype=np.float32
                ),
                cats,
            ),
        ],
        sparse_threshold=0,
    )
    return preprocess


In [ ]:
def build_model(name, params, columns, seed=42):
    """Construct an unfitted Random Forest, XGBoost or neural-network pipeline."""
    preprocess = make_preprocessor(columns)
    if name == "RandomForest":
        settings = dict(
            n_estimators=100,
            max_depth=16,
            min_samples_leaf=5,
            max_features=0.8,
            n_jobs=2,
            random_state=seed,
        )
        settings.update(params)
        estimator = RandomForestRegressor(**settings)
    elif name == "XGBoost":
        try:
            from xgboost import XGBRegressor
        except ImportError as exc:
            raise ImportError(
                "Install xgboost in this kernel; no agreed model is silently omitted."
            ) from exc
        settings = dict(
            objective="reg:squarederror",
            tree_method="hist",
            n_estimators=250,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=2,
            random_state=seed,
        )
        settings.update(params)
        estimator = XGBRegressor(**settings)
    elif name == "NeuralNetwork":
        # A feed-forward neural network. Target scaling and numerical feature scaling aid training.
        # Random internal early-stopping validation is disabled; outer folds are chronological.
        settings = dict(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            batch_size=512,
            max_iter=200,
            tol=1e-3,
            early_stopping=False,
            shuffle=True,
            random_state=seed,
        )
        settings.update(params)
        estimator = TransformedTargetRegressor(
            regressor=MLPRegressor(**settings), transformer=StandardScaler()
        )
    else:
        raise ValueError(name)
    return Pipeline([("preprocess", preprocess), ("model", estimator)])


In [ ]:
def columns_without(drop_groups):
    assert set(drop_groups) <= set(GROUPS)
    return [c for g, cols in GROUPS.items() if g not in drop_groups for c in cols]


In [ ]:
def add_losses(frame, scale):
    f = frame.copy()
    f["mase_scale"] = f.zone_id.map(scale)
    f["error"] = f.prediction - f.actual
    f["absolute_error"] = f.error.abs()
    f["squared_error"] = f.error**2
    f["scaled_absolute_error"] = f.absolute_error / f.mase_scale
    return f


In [ ]:
def metrics(frame, groups):
    """MASE is the mean zone-hour scaled error; zero training scales are undefined, not zero."""
    rows = []
    for keys, g in frame.groupby(groups, dropna=False, sort=False, observed=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        valid = g.actual.notna() & g.prediction.notna()
        v = g.loc[valid]
        row = dict(zip(groups, keys))
        row.update(
            MAE=v.absolute_error.mean(),
            RMSE=np.sqrt(v.squared_error.mean()),
            MASE=v.scaled_absolute_error.mean(),
            bias=v.error.mean(),
            n_scored=len(v),
            n_missing_target=int(g.actual.isna().sum()),
            n_missing_prediction=int(g.prediction.isna().sum()),
            n_undefined_mase=int(v.scaled_absolute_error.isna().sum()),
        )
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
def prediction_frame(data, meta, prediction, model, candidate, phase, actual=None):
    assert len(prediction) == len(meta) and np.isfinite(prediction).all()
    f = meta.copy()
    f["prediction"] = np.maximum(0, np.asarray(prediction, float))
    f["actual"] = (
        data.observations(meta) if actual is None else np.asarray(actual, float)
    )
    f["model"] = model
    f["candidate"] = candidate
    f["phase"] = phase
    return add_losses(f, data.scales(meta.origin.iloc[0]))


In [ ]:
def select_candidates(scores):
    """Choose one configuration per family using equal-weight mean validation MAE."""
    summary = scores.groupby(["candidate", "model"], as_index=False)[
        ["MAE", "RMSE", "MASE", "bias"]
    ].mean()
    assert np.isfinite(summary[["MAE", "RMSE"]]).all().all()
    best = (
        summary.sort_values(["MAE", "candidate"])
        .groupby("model", as_index=False)
        .first()
    )
    return summary, best


### Actual ARIMA fitting code
Each zone uses its full pre-origin history. Nonconvergence stops with a recorded error rather than silently substituting another model.

In [ ]:
def arima_forecast(data, origin, order):
    """Demand-only ARIMA per zone, full pre-origin history. Never substitutes a different model on error.
    This explicit nonseasonal baseline is compared with weekly-naive to expose missed seasonality.
    """
    forecast = np.empty((len(data.zone_ids), 168))
    logs = []
    train = data.panel[data.panel.index < pd.Timestamp(origin)]
    for i, zone in enumerate(data.zone_ids):
        y = train[zone].to_numpy(float)
        if np.nanmax(y) == np.nanmin(y):
            forecast[i] = max(0, float(np.nanmean(y)))
            logs.append(
                dict(zone_id=int(zone), status="constant_series", order=list(order))
            )
            continue
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                fitted = ARIMA(
                    y,
                    order=tuple(order),
                    trend="n" if order[1] > 0 else "c",
                    missing="none",
                ).fit(method_kwargs={"maxiter": 200})
            if not fitted.mle_retvals.get("converged", True):
                raise RuntimeError("ARIMA optimizer did not converge")
            values = np.asarray(fitted.forecast(168), float)
            assert np.isfinite(values).all()
            forecast[i] = np.maximum(0, values)
            logs.append(
                dict(
                    zone_id=int(zone),
                    status="ok",
                    order=list(order),
                    warnings=[str(w.message) for w in caught],
                )
            )
        except Exception as exc:
            logs.append(
                dict(
                    zone_id=int(zone),
                    status="failed",
                    order=list(order),
                    error=str(exc),
                )
            )
            json_write(
                Path(data.cfg["RESULTS_DIR"]) / "arima_failure.json",
                dict(origin=origin, records=logs),
            )
            raise RuntimeError(
                f"ARIMA failed for zone {zone}; see arima_failure.json. No substitution made."
            ) from exc
    return forecast.ravel(), logs


### Actual cross-validation training and tuning loop
Each candidate is fitted independently inside each expanding-window fold. Saved predictions are keyed by candidate, fold and feature set; compatible completed results can be resumed.

In [ ]:
def run_cv(data, candidates, drop_groups=(), tag="full"):
    """Sequential fits with the same examples, zones and hours. Save forecasts and fit warnings."""
    root = Path(data.cfg["RESULTS_DIR"])
    columns = columns_without(drop_groups)
    predictions = []
    fit_log = []
    signature = dict(
        candidates=candidates,
        drop_groups=list(drop_groups),
        manifest_sha256=hashlib.sha256(
            (root / "input_manifest.json").read_bytes()
        ).hexdigest(),
        helper_sha256=data.cfg["HELPER_SHA256"],
        implementation_sha256=data.cfg["IMPLEMENTATION_SHA256"],
    )
    receipt = root / f"{tag}_cv_signature.json"
    resume_allowed = receipt.exists()
    if resume_allowed:
        assert (
            json.loads(receipt.read_text()) == signature
        ), "CV inputs/code changed: choose a new run name."
    # Save the signature before fitting, so partial results can be resumed safely.
    json_write(receipt, signature)
    old_log = root / f"{tag}_fit_log.json"
    if old_log.exists():
        fit_log = json.loads(old_log.read_text())
    for fold, origin in enumerate(data.cfg["FOLDS"], 1):
        print(f"{tag}: fold {fold}/4, issue {origin}", flush=True)
        x, y, train_meta = data.training_examples(origin)
        future, meta = data.forecast_examples(origin)
        if tag == "full":
            zi = np.repeat(np.arange(len(data.zone_ids)), 168)
            naive = data.values_at(meta.target_datetime - pd.Timedelta(hours=168), zi)
            assert np.isfinite(
                naive
            ).all(), "Weekly-naive inputs unavailable: review common scoring mask"
            p = prediction_frame(
                data, meta, naive, "WeeklyNaive", "weekly_naive", "validation"
            )
            p["fold"] = fold
            p["feature_set"] = tag
            predictions.append(p)
        for candidate in candidates:
            started = time.perf_counter()
            name = candidate["model"]
            identifier = candidate["id"]
            cached = root / f"{tag}_{identifier}_fold{fold}_forecasts.parquet"
            if resume_allowed and cached.exists():
                saved = pd.read_parquet(cached)
                assert len(saved) == len(meta)
                assert (
                    saved.zone_id.to_numpy().tolist()
                    == meta.zone_id.to_numpy().tolist()
                )
                assert (
                    saved.target_datetime.to_numpy().tolist()
                    == meta.target_datetime.to_numpy().tolist()
                )
                predictions.append(saved)
                print("Reusing", identifier, "fold", fold, flush=True)
                continue
            if name == "ARIMA":
                predicted, logs = arima_forecast(
                    data, origin, candidate["params"]["order"]
                )
                json_write(root / f"{tag}_{identifier}_fold{fold}_arima_log.json", logs)
                rows = int((data.panel.index < pd.Timestamp(origin)).sum()) * len(
                    data.zone_ids
                )
                warn = []
            else:
                model = build_model(
                    name, candidate["params"], columns, data.cfg["SEED"]
                )
                with warnings.catch_warnings(record=True) as records:
                    warnings.simplefilter("always")
                    model.fit(x[columns], y)
                warn = [str(w.message) for w in records]
                predicted = model.predict(future[columns])
                rows = len(y)
                del model
            p = prediction_frame(data, meta, predicted, name, identifier, "validation")
            p["fold"] = fold
            p["feature_set"] = tag
            p.to_parquet(
                root / f"{tag}_{identifier}_fold{fold}_forecasts.parquet", index=False
            )
            predictions.append(p)
            fit_log.append(
                dict(
                    feature_set=tag,
                    fold=fold,
                    origin=origin,
                    candidate=identifier,
                    model=name,
                    training_examples=rows,
                    seconds=time.perf_counter() - started,
                    warnings=warn,
                )
            )
            json_write(root / f"{tag}_fit_log.json", fit_log)
        train_meta.assign(target_year=train_meta.target_datetime.dt.year).groupby(
            "target_year"
        ).size().to_csv(root / f"{tag}_fold{fold}_training_year_counts.csv")
        del x, y, future
    all_predictions = pd.concat(predictions, ignore_index=True)
    all_predictions.to_parquet(root / f"{tag}_cv_forecasts.parquet", index=False)
    scores = metrics(all_predictions, ["feature_set", "candidate", "model", "fold"])
    scores.to_csv(root / f"{tag}_cv_metrics.csv", index=False)
    json_write(receipt, signature)
    return all_predictions, scores


### SHAP and final-fit implementations
SHAP uses training backgrounds and held-out validation rows. Final fitting saves forecasts before test labels are read. These functions are defined here but never called in review/features mode.

In [ ]:
def shap_fold(data, candidate, origin, fold, drop_groups=()):
    """Explain held-out rows using training-only background; regroup dummy columns."""
    import shap

    columns = columns_without(drop_groups)
    x, y, _ = data.training_examples(origin)
    future, meta = data.forecast_examples(origin)
    model = build_model(
        candidate["model"], candidate["params"], columns, data.cfg["SEED"]
    )
    model.fit(x[columns], y)
    rng = np.random.default_rng(data.cfg["SEED"] + fold)
    chosen = rng.choice(
        len(future), min(data.cfg["SHAP_ROWS"], len(future)), replace=False
    )
    background = rng.choice(
        len(x), min(data.cfg["SHAP_BACKGROUND"], len(x)), replace=False
    )
    prep = model.named_steps["preprocess"]
    terminal = model.named_steps["model"]
    explain_x = prep.transform(future.iloc[chosen][columns])
    background_x = prep.transform(x.iloc[background][columns])
    names = prep.get_feature_names_out()
    if candidate["model"] in ("RandomForest", "XGBoost"):
        explainer = shap.TreeExplainer(
            terminal,
            data=background_x,
            feature_perturbation="interventional",
            model_output="raw",
        )
        explanation = explainer(explain_x, check_additivity=True)
    else:
        # Permutation SHAP explains the NN in original pickup units, including target inverse scaling.
        explainer = shap.Explainer(
            terminal.predict,
            shap.maskers.Independent(background_x),
            algorithm="permutation",
            feature_names=names,
            seed=data.cfg["SEED"],
        )
        explanation = explainer(explain_x, max_evals=2 * len(names) + 1)
    values = np.asarray(explanation.values)
    assert values.shape == explain_x.shape
    grouped = {name: np.zeros(len(chosen)) for name in columns}
    for i, encoded in enumerate(names):
        name = encoded.split("__", 1)[1]
        if name.startswith("missingindicator_"):
            name = name[len("missingindicator_") :]
        matches = [
            c for c in columns if name == c or (c in CAT and name.startswith(c + "_"))
        ]
        assert matches, f"Unmapped encoded feature: {encoded}"
        grouped[max(matches, key=len)] += values[:, i]
    contributions = pd.DataFrame(grouped)
    out = pd.concat(
        [meta.iloc[chosen].reset_index(drop=True), contributions.add_prefix("shap__")],
        axis=1,
    )
    out["fold"] = fold
    reconstructed = np.asarray(explanation.base_values).reshape(-1) + values.sum(axis=1)
    assert np.allclose(
        terminal.predict(explain_x), reconstructed, rtol=1e-3, atol=0.1
    ), "SHAP must reconstruct raw output"
    ranking = pd.DataFrame(
        dict(
            feature=columns,
            mean_abs_shap=[np.abs(grouped[c]).mean() for c in columns],
            fold=fold,
        )
    )
    ranking["group"] = ranking.feature.map(
        {c: g for g, cols in GROUPS.items() for c in cols}
    )
    root = Path(data.cfg["RESULTS_DIR"])
    out.to_parquet(root / f'shap_{candidate["id"]}_fold{fold}.parquet', index=False)
    ranking.to_csv(
        root / f'shap_{candidate["id"]}_fold{fold}_importance.csv', index=False
    )
    # Feature values accompany contributions for dependence plots.
    values_frame = future.iloc[chosen][columns].reset_index(drop=True)
    values_frame["fold"] = fold
    values_frame.to_parquet(
        root / f'shap_{candidate["id"]}_fold{fold}_feature_values.parquet', index=False
    )
    return ranking, out, values_frame


In [ ]:
def fit_final(data, selected, drop_groups):
    import joblib

    root = Path(data.cfg["RESULTS_DIR"])
    origin = data.cfg["TEST_ORIGIN"]
    assert (root / "frozen_experiment.json").exists()
    x, y, _ = data.training_examples(origin)
    future, meta = data.forecast_examples(origin)
    forecasts = []
    zi = np.repeat(np.arange(len(data.zone_ids)), 168)
    naive = data.values_at(meta.target_datetime - pd.Timedelta(hours=168), zi)
    forecasts.append(
        prediction_frame(
            data,
            meta,
            naive,
            "WeeklyNaive",
            "weekly_naive",
            "test",
            actual=np.full(len(meta), np.nan),
        )
    )
    for candidate in selected:
        if candidate["model"] == "ARIMA":
            pred, logs = arima_forecast(data, origin, candidate["params"]["order"])
            json_write(root / "final_arima_log.json", logs)
        else:
            columns = columns_without(drop_groups.get(candidate["id"], []))
            model = build_model(
                candidate["model"], candidate["params"], columns, data.cfg["SEED"]
            )
            model.fit(x[columns], y)
            pred = model.predict(future[columns])
            joblib.dump(model, root / f'final_{candidate["id"]}.joblib', compress=3)
            del model
        forecasts.append(
            prediction_frame(
                data,
                meta,
                pred,
                candidate["model"],
                candidate["id"],
                "test",
                actual=np.full(len(meta), np.nan),
            )
        )
    result = pd.concat(forecasts, ignore_index=True)
    result.to_parquet(root / "final_week_forecasts_unscored.parquet", index=False)
    return result


In [ ]:
def load_test_targets(data):
    """Call only after freezing. No test file -> no test scores; never fabricate labels."""
    import duckdb

    folder = Path(data.cfg["TEST_PICKUP_DIR"])
    if not list(folder.glob("*.parquet")) or not (folder / "_SUCCESS").exists():
        return (
            None,
            f"Test scoring unavailable: missing complete {folder.name}. Forecasts are saved without labels.",
        )
    db = duckdb.connect()
    db.execute("SET threads=1")
    db.execute("SET memory_limit='256MB'")
    observed, _ = read_month(folder, 2026, 1, db, allow_partial=True)
    db.close()
    times = pd.date_range(data.cfg["TEST_ORIGIN"], periods=168, freq="h")
    selected = observed[observed.timestamp.isin(times)]
    assert times.isin(
        selected.timestamp.unique()
    ).all(), "Test week has missing citywide hours"
    unknown = (
        selected.loc[~selected.zone_id.isin(data.zone_ids), "zone_id"].unique().tolist()
    )
    if unknown:
        json_write(
            Path(data.cfg["RESULTS_DIR"]) / "test_unseen_zones.json",
            dict(unseen_zone_ids=unknown),
        )
        warnings.warn(
            "Test zones outside the training-defined universe are reported separately."
        )
    wide = (
        selected.pivot(index="timestamp", columns="zone_id", values="pickups")
        .reindex(index=times, columns=data.zone_ids)
        .fillna(0)
    )
    return (
        wide.to_numpy(float).T.ravel(),
        "Test coverage complete for frozen eligible zones.",
    )


In [ ]:
def diagnostic_labels(frame, data):
    """Observed weather labels errors retrospectively; it never enters forecast features."""
    f = frame.copy()
    t = pd.DatetimeIndex(f.target_datetime)
    f["hour"] = t.hour
    f["weekday"] = t.day_name()
    f["date"] = t.normalize()
    h = holiday_features(t)
    f["holiday"] = np.where(h.is_holiday.to_numpy() == 1, "Holiday", "Other")
    f["borough"] = f.zone_id.map(data.static.borough)
    f["airport"] = np.where(f.zone_id.isin([1, 132, 138]), "Airport", "Other")
    f["demand_band"] = pd.cut(
        f.actual,
        [-np.inf, 0, 10, 50, 200, np.inf],
        labels=["Zero", "1-10", "11-50", "51-200", "Above 200"],
    )
    f["weather_location_id"] = f.zone_id.map(
        dict(zip(data.zone_ids, data.weather_locations))
    )
    w = data.weather[
        [
            "weather_location_id",
            "timestamp",
            "precipitation",
            "snowfall",
            "temperature_2m",
        ]
    ]
    f = f.merge(
        w,
        left_on=["weather_location_id", "target_datetime"],
        right_on=["weather_location_id", "timestamp"],
        how="left",
        validate="many_to_one",
    )
    f["weather_condition"] = np.select(
        [f.snowfall.gt(0), f.precipitation.gt(0), f.precipitation.notna()],
        ["Snow", "Rain", "Dry"],
        default="Missing weather",
    )
    f["temperature_band"] = pd.cut(
        f.temperature_2m,
        [-np.inf, 0, 10, 20, 30, np.inf],
        labels=["Below 0C", "0-10C", "10-20C", "20-30C", "Above 30C"],
    )
    return f


In [ ]:
# Bind the visible implementations so every later stage uses the code shown above.
# The external helper continues to provide only the static dictionary and plot builders.
zf.ForecastData = ForecastData
zf.build_model = build_model
zf.columns_without = columns_without
zf.add_losses = add_losses
zf.metrics = metrics
zf.prediction_frame = prediction_frame
zf.select_candidates = select_candidates
zf.arima_forecast = arima_forecast
zf.run_cv = run_cv
zf.shap_fold = shap_fold
zf.fit_final = fit_final
zf.load_test_targets = load_test_targets
zf.diagnostic_labels = diagnostic_labels


## Create the experiment output folder
Saved code/data fingerprints prevent accidentally mixing results from different feature implementations or inputs.

In [ ]:
# Keep each run isolated and stop if its saved settings no longer match.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CFG = {
    name: globals()[name]
    for name in [
        "DATA_DIR",
        "WEATHER_DIR",
        "ZONE_DIR",
        "RESULTS_DIR",
        "TEST_PICKUP_DIR",
        "SEED",
        "MAX_TRAIN_EXAMPLES",
        "SHAP_ROWS",
        "SHAP_BACKGROUND",
        "TRAIN_START",
        "TRAIN_END",
        "TEST_ORIGIN",
        "FOLDS",
    ]
}
# Hash saved implementation cells, not display outputs or mode switches.
# Save notebook edits before running so a changed implementation gets a new fingerprint.
notebook_document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
implementation_source = "\n".join(
    "".join(cell["source"])
    for cell in notebook_document["cells"]
    if cell.get("metadata", {}).get("role") == "implementation"
)
IMPLEMENTATION_SHA256 = hashlib.sha256(implementation_source.encode()).hexdigest()
CFG.update(
    {
        "FEATURE_DIR": FEATURE_DIR,
        "IMPLEMENTATION_SHA256": IMPLEMENTATION_SHA256,
        "HELPER_SHA256": hashlib.sha256(
            (HELPER_DIR / "zonal_forecasting.py").read_bytes()
        ).hexdigest(),
    }
)
serial_config = json.loads(json.dumps(CFG, default=str))
serial_config["n_candidates_per_model"] = N_CANDIDATES_PER_MODEL
serial_config["max_ablation_groups"] = MAX_ABLATION_GROUPS
serial_config["helper_sha256"] = hashlib.sha256(
    (HELPER_DIR / "zonal_forecasting.py").read_bytes()
).hexdigest()
config_file = RESULTS_DIR / "experiment_config.json"
if config_file.exists():
    assert (
        json.loads(config_file.read_text()) == serial_config
    ), "Settings changed: choose a new RUN_NAME."
else:
    zf.json_write(config_file, serial_config)
zf.json_write(RESULTS_DIR / "environment_versions.json", versions)


In [ ]:
# Use one consistent save/display function for every comparison and diagnostic plot.
plt.rcParams.update(
    {"figure.figsize": (12, 5), "axes.spines.top": False, "axes.spines.right": False}
)
COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#555555"]


def finish(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(RESULTS_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()


## 3. Read and audit the modeling data
Require all 60 monthly aggregates for 2021–2025, `_SUCCESS`, matching earlier-year contracts, unique zone/date/hour keys, positive counts and full non-DST citywide hours. Only within audited complete months do absent sparse zone-hour keys count as zero. Missing citywide hours are never silently filled.

Zones must have positive activity before the earliest validation fold; keep the same zone set across models/folds/test. Unknown geometries remain with `Unknown` borough and `zone_unmapped=1`. Their weather uses an explicitly reported Manhattan city proxy, not an invented borough. Export exclusions/mapping audits.

Nonexistent spring and combined autumn clock hours are excluded as labels and MASE pairs. Missing lag values remain missing and use train-only imputation. Recent rolling summaries allow a small number of missing DST hours and use pre-origin observations only. The six 2019 anomalous dates are outside this training scope.

The January 1–7, 2026 test-week aggregate has now been prepared from the raw January source using the existing 02-series cleaning rules. `TEST_PICKUP_DIR` points to its explicitly partial-week folder; it is not a full January aggregate. Its 19-column schema preserves the existing pickup metric definitions. Cleaning, key/date checks and cleaned-raw count reconciliations are recorded in `_evaluation_validation.json`. Only final evaluation cells read its zone-hour labels.

In [ ]:
# Loading/auditing is separate from fitting. Leave its switch off for code review.
data = None

if RUN_DATA_AUDIT:
    data = zf.ForecastData(CFG)
    display(pd.DataFrame(data.manifest)[["year", "month", "parts", "rows"]])
    display(pd.read_csv(RESULTS_DIR / "zone_eligibility.csv").head())
    print("Eligible zones:", len(data.zone_ids))
    print("Test-week aggregate exists:", TEST_PICKUP_DIR.exists())
else:
    print("Data loading is disabled. No training has started.")


### Check temporal coverage
Blue bands mark the four held-out validation weeks. Missing or combined DST hours remain explicit.

In [ ]:
# Plot the observed citywide total only after the data audit has been enabled.
if data is not None:
    city = data.panel.sum(axis=1, min_count=1)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(city.index, city, color="#444444", linewidth=0.35, alpha=0.6)

    # Shading illustrates validation periods; it does not remove later training data.
    for origin in FOLDS:
        start = pd.Timestamp(origin)
        ax.axvspan(start, start + pd.Timedelta(days=7), color="#0072B2", alpha=0.3)

    ax.set(title="Training coverage and validation weeks", ylabel="City pickups/hour")
    finish(fig, "06_training_coverage_and_folds")


## Feature data: generate, save and inspect
Run in `features` mode to stop after data preparation. Run in `full` mode to continue into training. Each origin folder contains `train_features.parquet`, `train_targets_and_metadata.parquet`, `forecast_features.parquet`, `forecast_metadata.parquet` and a quality/compatibility manifest.

In [ ]:
if RUN_FEATURE_EXPORT:
    assert data is not None
    for origin in FOLDS:
        export_feature_bundle(data, origin, include_validation_labels=True)
    # Prepare final training/future features without reading January test targets.
    export_feature_bundle(data, TEST_ORIGIN, include_validation_labels=False)
    print("Feature datasets saved under:", FEATURE_DIR)

    first_folder = feature_bundle_folder(data, FOLDS[0])
    preview = pd.read_parquet(first_folder / "train_features.parquet")
    labels = pd.read_parquet(first_folder / "train_targets_and_metadata.parquet")
    display(labels.head())
    with pd.option_context("display.max_rows", None):
        display(preview.head().T)
    del preview, labels
else:
    print(
        "No feature data generated in review mode. Choose features or full in the first cell."
    )


## 4. Feature engineering, encoding and fold definitions
The candidate set has 40 predictors: the previously discussed 39 plus `zone_unmapped`. Target timestamps and pickup labels are metadata, never predictors. Encoded dimensionality depends on training categories.

- Hour 0–23; weekday Monday=0; month 1–12; hour-of-week=weekday×24+hour. Cyclical hour/annual encodings connect endpoints. Initial redundancy can be reduced through validated ablation.
- US federal actual/observed holidays, holiday names/distances and a predefined December24–January2 period. This is not a school/event calendar.
- Demand at 168/336/504 hours before the TARGET; four-week same-hour mean also uses lag672. Every lag precedes the origin for a 168-hour forecast.
- Origin-based mean7d/mean28d/std28d are trips/hour; trend ratio is latest7d mean / preceding7d mean. These summaries remain fixed throughout the forecast week. Zero denominators stay missing.
- Weather predictors are pre-origin same-month/hour means with all-past fallback if that calendar group has not occurred yet. Units: Celsius, percent humidity/cloud, m/s wind, hPa pressure, mm precipitation/rain, cm snow. Provider precipitation refers to the preceding hour.

In [ ]:
# Export the same feature definitions shown above for machine-readable documentation.
dictionary = zf.feature_dictionary()
dictionary.to_csv(RESULTS_DIR / "feature_dictionary.csv", index=False)

# Grouped counts provide a compact check that no feature family was omitted.
display(dictionary.groupby("group", sort=False).size().rename("predictors").to_frame())
print("Total predictors before one-hot encoding:", len(dictionary))


### Chronological cross-validation schedule
All zones use identical boundaries. Each validation block contains 168 forecast hours, and its labels are excluded from that fold’s training data.

In [ ]:
# End boundaries are exclusive: training stops exactly when validation starts.
fold_table = pd.DataFrame(
    {
        "fold": range(1, 5),
        "train_start": TRAIN_START,
        "validation_start": pd.to_datetime(FOLDS),
    }
)
fold_table["train_end_exclusive"] = fold_table.validation_start
fold_table["validation_end_exclusive"] = fold_table.validation_start + pd.Timedelta(
    days=7
)

display(fold_table)
fold_table.to_csv(RESULTS_DIR / "fold_definitions.csv", index=False)


### Inspect engineered features before training
The transposed preview lists all 40 predictors as rows, rather than hiding them in a very wide table.

In [ ]:
# Constructing features does not fit a model or inspect validation labels.
if data is not None:
    example_x, example_meta = data.forecast_examples(FOLDS[0])
    display(example_meta.head())
    with pd.option_context("display.max_rows", None):
        display(example_x.head().T)


## 5. Evaluation metrics
Error = prediction − observation. MAE=mean absolute error; RMSE=square root of mean squared error; bias=mean signed error. For each zone, MASE divides absolute errors by that zone's training-only mean absolute 168-hour demand difference. Overall MASE averages zone-hour scaled errors. Zero training scales give undefined MASE, reported separately.

Select settings with equal-weight mean MAE over the four folds. Inspect RMSE/MASE/bias, fold variability and fit warnings. Evaluate identical zone-hours across models. Clip predictions at zero without rounding; SHAP explains pre-clipping outputs.

Random minibatch shuffling within neural-network TRAINING rows is permitted, but random internal early-stopping validation is disabled. The small candidate grid makes runtime explicit. ARIMA failure/nonconvergence stops the experiment with a zone/order log instead of silently substituting a different model.

## Model training and randomized hyperparameter search
Six reproducibly sampled configurations per ML family are compared on the same four folds. This is a bounded randomized search across multiple meaningful parameters, not only two manually chosen settings. MAE selects the configuration; RMSE, MASE and bias remain supporting metrics.

In [ ]:
# Multiple parameters are varied for each ML family. No fitting occurs here.
SEARCH_SPACES = {
    "RandomForest": {
        "n_estimators": [100, 200],
        "max_depth": [12, 20],
        "min_samples_leaf": [2, 8],
        "max_features": [0.6, 0.9],
    },
    "XGBoost": {
        "n_estimators": [200, 400],
        "max_depth": [3, 6, 9],
        "learning_rate": [0.03, 0.1],
        "subsample": [0.8],
        "colsample_bytree": [0.8],
    },
    "NeuralNetwork": {
        "hidden_layer_sizes": [(64, 32), (128, 64)],
        "alpha": [0.0001, 0.01],
        "learning_rate_init": [0.001, 0.003],
        "max_iter": [200],
    },
}
CANDIDATES = []
for family_index, (family, space) in enumerate(SEARCH_SPACES.items()):
    sampled = ParameterSampler(
        space, n_iter=N_CANDIDATES_PER_MODEL, random_state=SEED + family_index
    )
    for number, parameters in enumerate(sampled, 1):
        # Normalize tuples to JSON-compatible lists for stable checkpoints.
        parameters = json.loads(json.dumps(parameters))
        CANDIDATES.append(
            {"id": f"{family}_{number:02d}", "model": family, "params": parameters}
        )
for order in [(1, 0, 0), (1, 1, 1), (2, 1, 1)]:
    CANDIDATES.append(
        {
            "id": "arima_" + "".join(map(str, order)),
            "model": "ARIMA",
            "params": {"order": list(order)},
        }
    )

grid_path = RESULTS_DIR / "candidate_grid.json"
if grid_path.exists():
    assert (
        json.loads(grid_path.read_text()) == CANDIDATES
    ), "Use a new RUN_NAME for a changed grid."
else:
    json_write(grid_path, CANDIDATES)
display(pd.DataFrame(CANDIDATES))
print("Candidates:", len(CANDIDATES), "x", len(FOLDS), "validation folds")


### Run the actual tuning stage
The `run_cv` implementation above fits every candidate within each fold and computes MAE, RMSE, MASE and bias. Training matrices are read from the feature bundles exported earlier. Completed matching fold predictions are reused; incomplete work is refitted. No random train/test split is used.

In [ ]:
# Fit/tune only when explicitly enabled; otherwise read a compatible saved run.
cv_predictions, cv_scores = None, None
if RUN_CV:
    assert data is not None, "Run data audit first."
    assert not missing, "Install XGBoost and SHAP before the full experiment."
    cv_predictions, cv_scores = zf.run_cv(data, CANDIDATES)
    display(
        pd.DataFrame(json.loads((RESULTS_DIR / "full_fit_log.json").read_text()))[
            ["model", "fold", "seconds", "warnings"]
        ]
    )
elif (RESULTS_DIR / "full_cv_forecasts.parquet").exists():
    signature = json.loads((RESULTS_DIR / "full_cv_signature.json").read_text())
    assert (
        signature["helper_sha256"] == serial_config["helper_sha256"]
    ), "Cached CV code differs: use a new run."
    if data is not None:
        assert (
            signature["manifest_sha256"]
            == hashlib.sha256(
                (RESULTS_DIR / "input_manifest.json").read_bytes()
            ).hexdigest()
        ), "Source data changed: use a new run."
    assert (
        signature["implementation_sha256"] == IMPLEMENTATION_SHA256
    ), "Saved inline code differs: use a new run."
    cv_predictions = pd.read_parquet(RESULTS_DIR / "full_cv_forecasts.parquet")
    cv_scores = pd.read_csv(RESULTS_DIR / "full_cv_metrics.csv")
else:
    print("Cross-validation pending: enable RUN_CV when ready.")


### Compare validation scores
Select configurations by mean MAE across folds. RMSE, MASE and bias provide complementary checks; error bars below describe variation between validation weeks.

In [ ]:
# Summarize candidates and identify one validation-selected configuration per family.
summary, best_by_model = None, None

if cv_scores is not None:
    summary, best_by_model = zf.select_candidates(cv_scores)
    selected_scores = cv_scores[cv_scores.candidate.isin(best_by_model.candidate)]
    display(summary.sort_values("MAE"))
    summary.to_csv(RESULTS_DIR / "cv_candidate_summary.csv", index=False)


In [ ]:
# Compare all four metrics on separate axes because their units differ.
if cv_scores is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))

    for ax, metric in zip(axes.flat, ["MAE", "RMSE", "MASE", "bias"]):
        grouped = selected_scores.groupby("model")[metric]
        means = grouped.mean()
        ax.bar(
            means.index,
            means,
            yerr=grouped.std().fillna(0),
            color=COLORS[: len(means)],
            capsize=3,
        )
        ax.tick_params(axis="x", rotation=25)
        ax.set(title=f"Validation {metric}: mean and fold SD", ylabel=metric)
        if metric == "bias":
            ax.axhline(0, color="black", linewidth=0.7)

    finish(fig, "06_cv_model_metrics")


In [ ]:
# A model with good average error may still fail in an individual season.
if cv_scores is not None:
    fig, ax = plt.subplots(figsize=(12, 4))
    for name, group in selected_scores.groupby("model"):
        ax.plot(group.fold, group.MAE, marker="o", label=name)

    ax.set(
        xticks=range(1, 5),
        xlabel="Chronological fold",
        ylabel="MAE",
        title="Validation stability across weeks",
    )
    ax.legend()
    finish(fig, "06_cv_fold_stability")


## 7. SHAP explanations on validation data
Explain the leading validation ML candidate, refitted separately in each fold with a training-only background. Tree models use Tree SHAP; the neural network uses permutation SHAP in original pickup units. Sample sizes bound runtime. Dummy contributions are summed into their original variable BEFORE taking mean absolute SHAP.

Inspect global importance, fold stability and dependence plots. SHAP is not a significance test; correlated features can share importance. A low rank alone never triggers automatic removal.

In [ ]:
# Choose the leading ML candidate using validation MAE, never test performance.
shap_ranking, shap_contributions, shap_inputs = None, None, None
best_ml_candidate = None

if best_by_model is not None:
    ml_results = best_by_model[~best_by_model.model.isin(["ARIMA", "WeeklyNaive"])]
    leading = ml_results.sort_values("MAE").iloc[0]
    best_ml_candidate = next(
        candidate for candidate in CANDIDATES if candidate["id"] == leading.candidate
    )


In [ ]:
# SHAP refits the selected candidate within each fold, so keep this stage disabled.
# Its background rows come from training; its explained rows come from validation.
if RUN_SHAP:
    assert data is not None and best_ml_candidate is not None
    rankings, contributions, inputs = [], [], []

    for fold, origin in enumerate(FOLDS, 1):
        importance, effects, values = zf.shap_fold(
            data, best_ml_candidate, origin, fold
        )
        rankings.append(importance)
        contributions.append(effects)
        inputs.append(values)

    shap_ranking = pd.concat(rankings, ignore_index=True)
    shap_contributions = pd.concat(contributions, ignore_index=True)
    shap_inputs = pd.concat(inputs, ignore_index=True)
    shap_ranking.to_csv(RESULTS_DIR / "shap_validation_importance.csv", index=False)
else:
    print("SHAP computation is disabled; no model has been fitted by this stage.")


In [ ]:
# Aggregate dummy-variable contributions back to original feature names.
if shap_ranking is not None:
    top = (
        shap_ranking.groupby("feature").mean_abs_shap.mean().nlargest(20).sort_values()
    )
    fig, ax = plt.subplots(figsize=(10, 7))
    top.plot.barh(ax=ax, color="#0072B2")
    ax.set(
        title="Validation SHAP importance: original variables",
        xlabel="Mean absolute contribution, trips/hour",
    )
    finish(fig, "06_shap_global_importance")


In [ ]:
# Check whether a feature's apparent contribution is stable across validation weeks.
if shap_ranking is not None:
    matrix = shap_ranking.pivot(
        index="feature", columns="fold", values="mean_abs_shap"
    ).loc[top.index]
    fig, ax = plt.subplots(figsize=(8, 8))
    image = ax.imshow(matrix, aspect="auto", cmap="Blues")
    ax.set(
        yticks=range(len(matrix)),
        yticklabels=matrix.index,
        xticks=range(4),
        xticklabels=range(1, 5),
        xlabel="Fold",
        title="SHAP stability",
    )
    fig.colorbar(image, ax=ax, label="Mean absolute SHAP")
    finish(fig, "06_shap_fold_heatmap")


In [ ]:
# Show the relationship between a numerical feature and its SHAP contribution.
if shap_ranking is not None:
    numerical_features = [
        name for name in reversed(top.index.tolist()) if name not in zf.CAT
    ]
    if numerical_features:
        feature = numerical_features[0]
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.scatter(
            shap_inputs[feature],
            shap_contributions["shap__" + feature],
            s=10,
            alpha=0.35,
            color="#0072B2",
        )
        ax.axhline(0, color="#D55E00")
        ax.set(
            xlabel=feature,
            ylabel="SHAP contribution",
            title="Validation feature effect",
        )
        finish(fig, "06_shap_feature_effect")


## 8. Check proposed feature removals
After reviewing SHAP, choose `ABLATION_GROUPS` in the first cell. Retrain the leading ML candidate across all four folds for each removal. Remove at most one tested group: mean MAE must not worsen, and no fold's MAE may worsen by more than 5%. This criterion is predefined, not chosen from test results. Combined removals require their own validation.

Only the candidate actually tested receives its accepted reduced features. Other model families retain their validated full features. If no removal is requested or passes, retain all features.

In full mode, an empty `ABLATION_GROUPS` selects the two lowest-contribution eligible groups from validation SHAP automatically. They are only removed if the explicit retraining criteria pass. This enables a sequential Run All without requiring an interactive pause.

In [ ]:
# Removing a feature is a new training experiment, not just deleting a SHAP bar.
drop_by_candidate = {}
ablation_predictions, ablation_scores = {}, {}
ablation_table = None

if RUN_ABLATION:
    assert data is not None and best_ml_candidate is not None
    assert shap_ranking is not None, "Review validation SHAP first."
    if not ABLATION_GROUPS:
        # Low SHAP contribution suggests candidates; actual removals still require retraining.
        allowed = [g for g in GROUPS if g not in {"zone_identity", "demand_lags"}]
        group_scores = shap_ranking.groupby("group").mean_abs_shap.sum()
        ABLATION_GROUPS = (
            group_scores.reindex(allowed)
            .sort_values()
            .head(MAX_ABLATION_GROUPS)
            .index.tolist()
        )
    print("Groups to test by ablation:", ABLATION_GROUPS)

    base = (
        cv_scores[cv_scores.candidate.eq(best_ml_candidate["id"])]
        .set_index("fold")
        .MAE.sort_index()
    )
    comparisons = []

    for group in ABLATION_GROUPS:
        assert group in zf.GROUPS
        predictions, scores = zf.run_cv(
            data, [best_ml_candidate], drop_groups=[group], tag="without_" + group
        )
        ablation_predictions[group] = predictions
        ablation_scores[group] = scores
        relative_change = scores.set_index("fold").MAE.sort_index() / base - 1

        # Accept only if average MAE does not worsen and no fold worsens over 5%.
        acceptable = scores.MAE.mean() <= base.mean() and relative_change.max() <= 0.05
        comparisons.append(
            {
                "removed_group": group,
                "mean_MAE": scores.MAE.mean(),
                "delta_MAE": scores.MAE.mean() - base.mean(),
                "worst_fold_relative_change": relative_change.max(),
                "acceptable": acceptable,
            }
        )

    ablation_table = pd.DataFrame(comparisons)
    display(ablation_table)
    ablation_table.to_csv(RESULTS_DIR / "feature_ablation_comparison.csv", index=False)


In [ ]:
# Negative bars mean improvement after removing that group.
if ablation_table is not None:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(ablation_table.removed_group, ablation_table.delta_MAE, color="#0072B2")
    ax.axhline(0, color="#D55E00")
    ax.set(
        title="Feature removal: change in validation MAE",
        ylabel="MAE change; negative is better",
    )
    finish(fig, "06_feature_ablation")

    # Carry forward at most ONE tested removal; do not combine untested changes.
    accepted = ablation_table[ablation_table.acceptable].sort_values("mean_MAE")
    if not accepted.empty:
        group = accepted.iloc[0].removed_group
        drop_by_candidate[best_ml_candidate["id"]] = [group]

print("Validation-supported removals:", drop_by_candidate)


## 9. Diagnose errors at different levels
Inspect spatial, temporal, forecast-horizon, activity, holiday and weather patterns. Export every subgroup's support count, missing-target count and undefined-MASE count. Small groups and a few unusual weeks cannot establish general causes. Diagnostic actual weather is kept separate from prediction features.

Use validation diagnostics to improve models. Final test diagnostics report the frozen experiment and must not become a tuning loop. Bias can cancel across zones, so examine signed maps and residual distributions, not only overall averages.


The plotting functions are grouped by purpose in the helper module. The short orchestration cell below shows which tables and figures are produced. Each plot helper returns a figure; `finish()` handles its display and saving.

In [ ]:
# Retain only validation-selected candidates; use reduced features only where tested.
diagnostic_predictions = None
if cv_predictions is not None and best_by_model is not None:
    diagnostic_predictions = cv_predictions[
        cv_predictions.candidate.isin(best_by_model.candidate)
    ].copy()
    for identifier, groups in drop_by_candidate.items():
        diagnostic_predictions = diagnostic_predictions[
            ~diagnostic_predictions.candidate.eq(identifier)
        ]
        diagnostic_predictions = pd.concat(
            [diagnostic_predictions, ablation_predictions[groups[0]]], ignore_index=True
        )


In [ ]:
def diagnose(predictions, prefix):
    """Export subgroup metrics and display consistent multi-level error diagnostics."""
    # Actual weather is attached AFTER prediction, for retrospective diagnostics only.
    labeled = zf.diagnostic_labels(predictions, data)
    by_zone = zf.metrics(labeled, ["model", "zone_id"])
    by_zone.to_csv(RESULTS_DIR / f"{prefix}_zone_metrics.csv", index=False)

    # Diagnose deterioration over the seven forecast days.
    by_day = zf.metrics(labeled, ["model", "forecast_day"])
    finish(zf.plot_horizon_errors(by_day), prefix + "_horizon_errors")

    # Each subgroup plot includes error size, bias and the observation count.
    dimensions = [
        "borough",
        "airport",
        "hour",
        "weekday",
        "holiday",
        "demand_band",
        "weather_condition",
        "temperature_band",
    ]
    for dimension in dimensions:
        table = zf.metrics(labeled, ["model", dimension])
        table.to_csv(RESULTS_DIR / f"{prefix}_{dimension}_metrics.csv", index=False)
        finish(
            zf.plot_subgroup_errors(table, dimension, COLORS), prefix + "_" + dimension
        )

    # Shared limits make maps comparable between models.
    mae_max = max(float(by_zone.MAE.max()), 0.01)
    bias_limit = max(float(by_zone.bias.abs().max()), 0.01)

    for model, records in labeled.groupby("model"):
        zone_table = by_zone[by_zone.model.eq(model)]
        figure = zf.plot_zone_errors(data.zones, zone_table, model, mae_max, bias_limit)
        finish(figure, prefix + "_" + model + "_zone_maps")

        # Unmapped zones keep their metrics even though they cannot appear on a map.
        unmapped = zone_table[~zone_table.zone_id.isin(data.zones.LocationID)]
        unmapped.to_csv(
            RESULTS_DIR / f"{prefix}_{model}_unmapped_zone_metrics.csv", index=False
        )
        finish(
            zf.plot_hourly_error_heatmap(records, model),
            prefix + "_" + model + "_hour_heatmap",
        )
        finish(
            zf.plot_residual_diagnostics(records, model, SEED),
            prefix + "_" + model + "_residuals",
        )
        finish(
            zf.plot_forecast_overlay(records, model), prefix + "_" + model + "_overlay"
        )

    return labeled


In [ ]:
# Diagnostic plots use saved predictions; they do not train another model.
validation_diagnostics = None
if data is not None and diagnostic_predictions is not None:
    validation_diagnostics = diagnose(diagnostic_predictions, "06_validation")


## 10. Freeze the specification before final evaluation
Select one configuration per family from validation and carry accepted removals only into the candidate where tested. Export the winner, features, parameters, sampling budget, zones, weather assumptions, versions, source manifest and helper checksum. Notebook 07 must use this specification unchanged.

Review convergence warnings before acknowledging them. Poor neural-network convergence should prompt revised training settings and revalidation in a NEW run, not an unqualified final result. An existing frozen specification cannot be overwritten with changed settings under the same run name. January 2026 pickup labels have not been read yet.

Fit warnings are always saved. Set `FAIL_ON_FIT_WARNINGS=True` for a strict review stop; otherwise the final report retains warnings alongside scores. Nonfinite predictions and actual fitting failures remain errors.

In [ ]:
# Prepare the validation-selected configuration, without reading test labels.
frozen = None
selected_candidates, selected_summary, winner = None, None, None
fit_warnings = []

if best_by_model is not None:
    selected_candidates = [
        candidate
        for candidate in CANDIDATES
        if candidate["id"] in best_by_model.candidate.tolist()
    ]
    updated_scores = cv_scores[cv_scores.candidate.isin(best_by_model.candidate)].copy()

    # Substitute ablation scores only for the candidate whose removal was validated.
    for identifier, groups in drop_by_candidate.items():
        updated_scores = updated_scores[~updated_scores.candidate.eq(identifier)]
        updated_scores = pd.concat(
            [updated_scores, ablation_scores[groups[0]]], ignore_index=True
        )

    selected_summary = updated_scores.groupby(["candidate", "model"], as_index=False)[
        ["MAE", "RMSE", "MASE", "bias"]
    ].mean()
    winner = selected_summary.sort_values("MAE").iloc[0]
    best_ml_row = (
        selected_summary[
            selected_summary.model.isin(["RandomForest", "XGBoost", "NeuralNetwork"])
        ]
        .sort_values("MAE")
        .iloc[0]
    )
    print("Best validated ML candidate:", best_ml_row.candidate)
    print("Best overall candidate, including baselines:", winner.candidate)
    display(selected_summary.sort_values("MAE"))

    # Review recorded fit/convergence warnings before accepting a specification.
    logs = json.loads((RESULTS_DIR / "full_fit_log.json").read_text())
    for group in ABLATION_GROUPS:
        log_path = RESULTS_DIR / f"without_{group}_fit_log.json"
        if group in ablation_scores and log_path.exists():
            logs.extend(json.loads(log_path.read_text()))
    fit_warnings = [record for record in logs if record["warnings"]]


In [ ]:
def make_frozen_specification():
    """Record settings needed to reproduce 06 and hold the future 07 experiment fixed."""
    # These hashes detect changes to the helper code or underlying training files.
    helper_hash = hashlib.sha256(
        (HELPER_DIR / "zonal_forecasting.py").read_bytes()
    ).hexdigest()
    manifest_hash = hashlib.sha256(
        (RESULTS_DIR / "input_manifest.json").read_bytes()
    ).hexdigest()

    features_by_candidate = {
        candidate["id"]: zf.columns_without(drop_by_candidate.get(candidate["id"], []))
        for candidate in selected_candidates
    }
    return {
        "config": serial_config,
        "selected_candidates": selected_candidates,
        "drop_by_candidate": drop_by_candidate,
        "selected_features": features_by_candidate,
        "validation_winner": winner.candidate,
        "validation_best_ml": best_ml_row.candidate,
        "validation_summary": selected_summary.to_dict("records"),
        "weather_assumption": "Past-only climatology; NOT actual future weather",
        "sample_policy": "Seeded bounded examples spanning all training years",
        "zone_ids": data.zone_ids.tolist(),
        "versions": versions,
        "helper_sha256": helper_hash,
        "manifest_sha256": manifest_hash,
        "fit_warnings_acknowledged": bool(ACKNOWLEDGE_FIT_WARNINGS),
    }


In [ ]:
# Freezing is allowed only after the validation and interpretation stages exist.
if best_by_model is not None:
    if fit_warnings and FAIL_ON_FIT_WARNINGS and not ACKNOWLEDGE_FIT_WARNINGS:
        print("Freeze pending: inspect fit logs and convergence warnings.")
    elif RUN_SHAP and shap_ranking is not None:
        if fit_warnings:
            print(
                "Fit warnings retained in the frozen report; review convergence before interpreting results."
            )
        frozen = make_frozen_specification()
        frozen["fit_warning_records"] = fit_warnings
        frozen["implementation_sha256"] = IMPLEMENTATION_SHA256
        path = RESULTS_DIR / "frozen_experiment.json"
        normalized = json.loads(json.dumps(frozen, default=str))

        # Never overwrite a different frozen specification under the same run name.
        if path.exists():
            assert (
                json.loads(path.read_text()) == normalized
            ), "Frozen specification changed: use a new RUN_NAME."
        else:
            zf.json_write(path, frozen)
        print("Frozen validation winner:", winner.candidate)
    else:
        print("Freeze pending: complete validation SHAP and optional ablation.")


## 11. Final-week forecast and held-out evaluation
Refit the selected configuration of each family using the full 2021–2025 period (ML sampling budget unchanged). Save the entire week's forecasts BEFORE reading any January pickup targets. Then score January1–7 if complete aggregates exist. Missing targets block scoring only, without triggering raw-data processing or fabricated metrics.

Use training-only MASE scales and the same four metrics/diagnostics as validation. Do not change model/features after inspecting this week. Notebook 07's five history windows are a preplanned separate experiment with fixed specifications.

In [ ]:
# This is the ONLY final fitting entry point; RUN_FINAL remains False for review.
test_forecasts, test_scores = None, None

if RUN_FINAL:
    assert frozen is not None and data is not None, "Freeze validation settings first."

    # Save all predictions before opening any January 2026 pickup labels.
    test_forecasts = zf.fit_final(
        data, frozen["selected_candidates"], frozen["drop_by_candidate"]
    )
else:
    print("Final model fitting is disabled.")


In [ ]:
# Test labels enter evaluation only after the forecast has already been saved.
if RUN_FINAL and test_forecasts is not None:
    actual, status = zf.load_test_targets(data)
    print(status)

    if actual is not None:
        scored_parts = []
        for candidate, records in test_forecasts.groupby("candidate", sort=False):
            records = records.copy()
            assert len(records) == len(actual)
            records["actual"] = actual
            scored_parts.append(zf.add_losses(records, data.scales(TEST_ORIGIN)))

        test_forecasts = pd.concat(scored_parts, ignore_index=True)
        test_scores = zf.metrics(test_forecasts, ["candidate", "model"])
        test_forecasts.to_parquet(
            RESULTS_DIR / "final_week_forecasts_scored.parquet", index=False
        )
        test_scores.to_csv(RESULTS_DIR / "final_test_metrics.csv", index=False)
        display(test_scores.sort_values("MAE"))
    else:
        zf.json_write(
            RESULTS_DIR / "test_evaluation_status.json",
            {"status": "blocked_missing_targets", "reason": status},
        )


In [ ]:
# Final comparison is reporting only; do not retune using these results.
if test_scores is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, metric in zip(axes.flat, ["MAE", "RMSE", "MASE", "bias"]):
        values = test_scores.set_index("model")[metric]
        values.plot.bar(ax=ax, color=COLORS[: len(values)])
        ax.set(title=f"Final held-out week: {metric}", ylabel=metric)
        if metric == "bias":
            ax.axhline(0, color="black", linewidth=0.6)
    finish(fig, "06_final_test_model_comparison")


In [ ]:
# Reuse identical diagnostic logic for validation and final test predictions.
test_diagnostics = None
if test_scores is not None:
    test_diagnostics = diagnose(test_forecasts, "06_final_test")


## 12. Results, status and limitations
- Data coverage, eligibility/exclusions, sparse-zero assumptions, unknown geometries, DST and test-label availability.
- Mean/variation of validation MAE, RMSE, MASE and bias, selected candidate per family, warnings and runtime.
- SHAP stability/correlation, feature-removal hypotheses and cross-validated retraining evidence.
- Error concentration by zone/borough, hour/weekday/holiday, forecast day, demand and weather, with support counts. Association with errors does not establish their cause.
- Frozen model/features and held-out results versus weekly-naive; no subsequent tuning on this test week.
- A single winter/holiday week does not establish year-round performance. Weather is coarse retrospective ERA5; FUTURE features here use past-only climatology, not actual future conditions. ARIMA is an initial nonseasonal baseline. Global ML and local ARIMA share information differently.
- Seeded ML sampling is resource-limited, explicit and spans all years. Results concern this training budget, not unlimited-data/tuning performance.
- Carry `frozen_experiment.json` into notebook 07; no training-history comparisons run here.


**Authored status:** no feature generation, training or tuning has been executed during this revision. The saved default is review mode. To produce real results, install required dependencies, set EXECUTION_MODE="full", save, and Run All. The best model means the best validated candidate within the configured search/sample budget.